In [1]:
from sympy.physics.mechanics import *
from sympy import symbols, atan, Matrix, solve
# Create generalized coordinates and speeds for this non-minimal realization
# q1, q2 = N.x and N.y coordinates of pendulum
# u1, u2 = N.x and N.y velocities of pendulum
q1, q2 = dynamicsymbols('q1:3')
q1d, q2d = dynamicsymbols('q1:3', level=1)
u1, u2 = dynamicsymbols('u1:3')
u1d, u2d = dynamicsymbols('u1:3', level=1)
L, m, g, t = symbols('L, m, g, t')

In [2]:
# Compose world frame
N = ReferenceFrame('N')
pN = Point('N*')
pN.set_vel(N, 0)

# A.x is along the pendulum
theta1 = atan(q2/q1)
A = N.orientnew('A', 'axis', [theta1, N.z])

In [3]:
# Locate the pendulum mass
P = pN.locatenew('P1', q1*N.x + q2*N.y)
pP = Particle('pP', P, m)

In [4]:
# Calculate the kinematic differential equations
kde = Matrix([q1d - u1,
              q2d - u2])
dq_dict = solve(kde, [q1d, q2d])

In [5]:
f_c = Matrix([P.pos_from(pN).magnitude() - L])
f_v = Matrix([P.vel(N).express(A).dot(A.x)])
f_v.simplify()

In [6]:
# Input the force resultant at P
R = m*g*N.x

In [7]:
# Derive the equations of motion using the KanesMethod class.
KM = KanesMethod(N, q_ind=[q2], u_ind=[u2], q_dependent=[q1],
                 u_dependent=[u1], configuration_constraints=f_c,
                 velocity_constraints=f_v, kd_eqs=kde)
(fr, frstar) = KM.kanes_equations([pP],[(P, R)])

In [9]:
# Set the operating point to be straight down, and non-moving
q_op = {q1: L, q2: 0}
u_op = {u1: 0, u2: 0}
ud_op = {u1d: 0, u2d: 0}
# Perform the linearization
A, B, inp_vec = KM.linearize(op_point=[q_op, u_op, ud_op], A_and_B=True,
                             new_method=True, simplify=True)

In [10]:
A

Matrix([
[   0, 1],
[-g/L, 0]])

In [11]:
B

Matrix(0, 0, [])